# Notebook 03d v2 — Bootstrap CIs on calibration metrics

**Purpose:** Quantify uncertainty in the calibration findings using bootstrap resampling over the test set. Addresses the TA's bootstrap CI request as it applies to calibration claims.

**Methodology:**
- B=1000 bootstrap resamples of the test set, stratified by class to preserve class proportions
- For each resample: recompute macro Brier (pre and post), macro ECE (pre and post), argmax flipping rate, accuracy delta, macro-F1 delta
- Report 95% CIs (2.5/97.5 percentile) on each metric and on the pre→post deltas
- Run across all three datasets × both methods × all 5-class models (binary models excluded for brevity; can be added if needed)

**What this answers:**
- Are the calibration improvements/degradations we observed statistically robust, or could they be test-set fluctuations?
- Is the dramatic UNSW Dirichlet argmax flipping (14-29%) a real effect or could it be much smaller?
- Does the NSL Dirichlet macro-F1 drop have a CI that excludes zero?

**Compute estimate:** ~30 minutes total across all datasets and models (B=1000 is fast for these metric computations).

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, shutil
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)

for f in ['.gitconfig', '.git-credentials']:
    src = f'/content/drive/MyDrive/XIDS_Research/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/root/{f}')
        if f == '.git-credentials':
            os.chmod(f'/root/{f}', 0o600)

!git pull
print(f'\n✓ Ready in: {os.getcwd()}')

Mounted at /content/drive
Already up to date.

✓ Ready in: /content/drive/MyDrive/XIDS_Research/xids-research


In [2]:
import numpy as np
import pandas as pd
import json
from pathlib import Path
from collections import Counter
from sklearn.metrics import brier_score_loss, accuracy_score, f1_score
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

SEED = 42
rng = np.random.default_rng(SEED)

B = 1000  # number of bootstrap resamples
ECE_N_BINS = 10

DATASETS = {
    'nsl_kdd_v2': {'probs_dir': 'predictions'},
    'unsw_nb15_v2': {'probs_dir': 'probabilities'},
    'cic_ids2017_v2': {'probs_dir': 'probabilities'},
}

MODELS_5CLASS = ['rf_5class_smote', 'xgb_5class_smote', 'dnn_5class_smote',
                 'rf_5class_cw', 'xgb_5class_cw', 'dnn_5class_cw']

METHODS = {
    'hybrid': '',           # calibrators/{dataset}/
    'dirichlet': '_dirichlet',  # calibrators/{dataset}_dirichlet/
}

print(f'Bootstrap iterations: B={B}')
print(f'Datasets: {list(DATASETS.keys())}')
print(f'Methods: {list(METHODS.keys())}')
print(f'Models: {MODELS_5CLASS}')

Bootstrap iterations: B=1000
Datasets: ['nsl_kdd_v2', 'unsw_nb15_v2', 'cic_ids2017_v2']
Methods: ['hybrid', 'dirichlet']
Models: ['rf_5class_smote', 'xgb_5class_smote', 'dnn_5class_smote', 'rf_5class_cw', 'xgb_5class_cw', 'dnn_5class_cw']


## 2. Helper functions

In [3]:
def expected_calibration_error(probs, labels, n_bins=10):
    """ECE on one-vs-rest binary probabilities."""
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    n = len(probs)
    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        mask = (probs >= lo) & (probs <= hi) if i == n_bins - 1 else (probs >= lo) & (probs < hi)
        if mask.sum() == 0:
            continue
        ece += (mask.sum() / n) * abs(probs[mask].mean() - labels[mask].mean())
    return float(ece)

def macro_brier(p, y, n_classes=5):
    """Macro Brier across classes (one-vs-rest)."""
    return float(np.mean([brier_score_loss((y == c).astype(int), p[:, c]) for c in range(n_classes)]))

def macro_ece(p, y, n_classes=5, n_bins=10):
    """Macro ECE across classes (one-vs-rest)."""
    return float(np.mean([expected_calibration_error(p[:, c], (y == c).astype(int), n_bins) for c in range(n_classes)]))

def compute_metrics(p_pre, p_post, y, n_classes=5):
    """Compute all metrics on a single resample."""
    pred_pre = p_pre.argmax(axis=1)
    pred_post = p_post.argmax(axis=1)
    return {
        'brier_pre': macro_brier(p_pre, y, n_classes),
        'brier_post': macro_brier(p_post, y, n_classes),
        'ece_pre': macro_ece(p_pre, y, n_classes),
        'ece_post': macro_ece(p_post, y, n_classes),
        'pct_flipped': 100.0 * (pred_pre != pred_post).mean(),
        'acc_pre': accuracy_score(y, pred_pre),
        'acc_post': accuracy_score(y, pred_post),
        'f1m_pre': f1_score(y, pred_pre, average='macro', zero_division=0),
        'f1m_post': f1_score(y, pred_post, average='macro', zero_division=0),
    }

def stratified_bootstrap_indices(y, rng_local):
    """Resample test indices stratified by class to preserve class proportions."""
    n = len(y)
    indices = np.empty(n, dtype=np.int64)
    pos = 0
    for c in np.unique(y):
        class_idx = np.where(y == c)[0]
        sampled = rng_local.choice(class_idx, size=len(class_idx), replace=True)
        indices[pos:pos + len(sampled)] = sampled
        pos += len(sampled)
    return indices

def bootstrap_ci(values, alpha=0.05):
    """Compute percentile CI (2.5/97.5 by default)."""
    lo = np.percentile(values, 100 * alpha / 2)
    hi = np.percentile(values, 100 * (1 - alpha / 2))
    return float(lo), float(hi), float(np.mean(values)), float(np.std(values))

print('✓ Helpers loaded')

✓ Helpers loaded


## 3. Bootstrap loop

In [ ]:
all_results = []

for dataset, ds_info in DATASETS.items():
    print(f'\n{"="*70}')
    print(f'Dataset: {dataset}')
    print('='*70)

    PROCESSED = Path(REPO) / 'data' / 'processed' / dataset
    PROBS_DIR = Path(REPO) / 'models' / dataset / ds_info['probs_dir']
    y_test = np.load(PROCESSED / 'y_test_5class.npy')

    for method_name, method_suffix in METHODS.items():
        CALIB_DIR = Path(REPO) / 'calibrators' / f'{dataset}{method_suffix}'

        if not CALIB_DIR.exists():
            print(f'  Skipping {method_name}: {CALIB_DIR} not found')
            continue

        print(f'\n  Method: {method_name}')

        for model_name in MODELS_5CLASS:
            p_pre_path = PROBS_DIR / f'{model_name}_test_proba.npy'
            p_post_path = CALIB_DIR / f'{model_name}_test_proba_calibrated.npy'

            if not p_pre_path.exists() or not p_post_path.exists():
                print(f'    {model_name}: missing files, skipping')
                continue

            p_pre = np.load(p_pre_path)
            p_post = np.load(p_post_path)

            # Point estimate (on full test set)
            point = compute_metrics(p_pre, p_post, y_test, n_classes=5)

            # Bootstrap resamples
            bs_metrics = {k: [] for k in point.keys()}
            bs_metrics['brier_delta'] = []
            bs_metrics['ece_delta'] = []
            bs_metrics['acc_delta'] = []
            bs_metrics['f1m_delta'] = []

            rng_local = np.random.default_rng(SEED)
            for b in range(B):
                idx = stratified_bootstrap_indices(y_test, rng_local)
                m = compute_metrics(p_pre[idx], p_post[idx], y_test[idx], n_classes=5)
                for k, v in m.items():
                    bs_metrics[k].append(v)
                bs_metrics['brier_delta'].append(m['brier_post'] - m['brier_pre'])
                bs_metrics['ece_delta'].append(m['ece_post'] - m['ece_pre'])
                bs_metrics['acc_delta'].append(m['acc_post'] - m['acc_pre'])
                bs_metrics['f1m_delta'].append(m['f1m_post'] - m['f1m_pre'])

            # Compute CIs for key metrics
            row = {
                'dataset': dataset,
                'method': method_name,
                'model': model_name,
                # Point estimates
                'pct_flipped_point': round(point['pct_flipped'], 4),
                'brier_delta_point': round(point['brier_post'] - point['brier_pre'], 5),
                'ece_delta_point': round(point['ece_post'] - point['ece_pre'], 5),
                'acc_delta_point': round(point['acc_post'] - point['acc_pre'], 5),
                'f1m_delta_point': round(point['f1m_post'] - point['f1m_pre'], 5),
            }

            for metric_label, metric_key in [
                ('pct_flipped', 'pct_flipped'),
                ('brier_delta', 'brier_delta'),
                ('ece_delta', 'ece_delta'),
                ('acc_delta', 'acc_delta'),
                ('f1m_delta', 'f1m_delta'),
            ]:
                lo, hi, mean, std = bootstrap_ci(bs_metrics[metric_key])
                row[f'{metric_label}_ci_lo'] = round(lo, 5)
                row[f'{metric_label}_ci_hi'] = round(hi, 5)
                row[f'{metric_label}_ci_width'] = round(hi - lo, 5)

            all_results.append(row)

            # Print summary line for this (dataset, method, model)
            print(f'    {model_name:<22} '
                  f'ΔBrier {row["brier_delta_point"]:+.4f} '
                  f'[{row["brier_delta_ci_lo"]:+.4f}, {row["brier_delta_ci_hi"]:+.4f}]  '
                  f'%flip {row["pct_flipped_point"]:.2f}% '
                  f'[{row["pct_flipped_ci_lo"]:.2f}, {row["pct_flipped_ci_hi"]:.2f}]')

print(f'\n✓ Bootstrap complete. {len(all_results)} (dataset, method, model) cells.')


Dataset: nsl_kdd_v2

  Method: hybrid
    rf_5class_smote        ΔBrier +0.0124 [+0.0119, +0.0129]  %flip 3.61% [3.35, 3.83]
    xgb_5class_smote       ΔBrier +0.0009 [+0.0007, +0.0012]  %flip 0.21% [0.16, 0.27]
    dnn_5class_smote       ΔBrier +0.0029 [+0.0025, +0.0033]  %flip 2.72% [2.53, 2.94]
    rf_5class_cw           ΔBrier +0.0087 [+0.0082, +0.0091]  %flip 4.74% [4.49, 5.01]
    xgb_5class_cw          ΔBrier -0.0003 [-0.0005, -0.0001]  %flip 0.26% [0.20, 0.32]
    dnn_5class_cw          ΔBrier +0.0050 [+0.0043, +0.0058]  %flip 7.35% [7.03, 7.71]

  Method: dirichlet
    rf_5class_smote        ΔBrier +0.0074 [+0.0072, +0.0076]  %flip 1.74% [1.57, 1.91]
    xgb_5class_smote       ΔBrier +0.0010 [+0.0008, +0.0012]  %flip 1.13% [0.99, 1.27]
    dnn_5class_smote       ΔBrier +0.0019 [+0.0015, +0.0023]  %flip 2.58% [2.39, 2.78]
    rf_5class_cw           ΔBrier +0.0046 [+0.0044, +0.0048]  %flip 0.75% [0.63, 0.86]
    xgb_5class_cw          ΔBrier +0.0022 [+0.0020, +0.0024]  %flip 1.

## 4. Save results to CSV

In [ ]:
df = pd.DataFrame(all_results)
TABLES_DIR = Path(REPO) / 'results' / 'tables'
TABLES_DIR.mkdir(parents=True, exist_ok=True)
csv_path = TABLES_DIR / 'calibration_bootstrap_cis.csv'
df.to_csv(csv_path, index=False)
print(f'✓ Saved: {csv_path}')
print(f'  Rows: {len(df)}')
print(f'  Columns: {list(df.columns)}')

## 5. Headline summary — does each claim survive bootstrap?

In [ ]:
def ci_excludes_zero(row, metric):
    """Returns True if CI for the delta excludes zero (statistically significant)."""
    lo = row[f'{metric}_ci_lo']
    hi = row[f'{metric}_ci_hi']
    return (lo > 0 and hi > 0) or (lo < 0 and hi < 0)

def summarize_claim(df, dataset, method, claim_name, metric, expected_direction):
    """Summarize a paper claim against bootstrap CIs.

    expected_direction: 'positive' (delta > 0), 'negative' (delta < 0), or 'any'
    """
    sub = df[(df['dataset'] == dataset) & (df['method'] == method)].copy()
    if len(sub) == 0:
        return f'{claim_name}: no data'

    n_total = len(sub)
    n_excludes_zero = sum(ci_excludes_zero(row, metric) for _, row in sub.iterrows())

    if expected_direction == 'positive':
        n_supports = sum(row[f'{metric}_ci_lo'] > 0 for _, row in sub.iterrows())
    elif expected_direction == 'negative':
        n_supports = sum(row[f'{metric}_ci_hi'] < 0 for _, row in sub.iterrows())
    else:
        n_supports = n_excludes_zero

    return f'{claim_name}: {n_supports}/{n_total} models support claim ({n_excludes_zero}/{n_total} CIs exclude zero)'

print('Headline calibration claims under bootstrap:\n')

# Claim 1: UNSW Dirichlet improves Brier on all 5-class models
print(summarize_claim(df, 'unsw_nb15_v2', 'dirichlet',
                      'UNSW Dirichlet Brier improvement (negative delta)',
                      'brier_delta', 'negative'))

# Claim 2: UNSW hybrid improves Brier on all 5-class models
print(summarize_claim(df, 'unsw_nb15_v2', 'hybrid',
                      'UNSW hybrid Brier improvement (negative delta)',
                      'brier_delta', 'negative'))

# Claim 3: UNSW Dirichlet substantial argmax flipping (>0)
print(summarize_claim(df, 'unsw_nb15_v2', 'dirichlet',
                      'UNSW Dirichlet argmax flipping is real (CI excludes 0)',
                      'pct_flipped', 'any'))

# Claim 4: NSL Dirichlet macro-F1 drop is real
print(summarize_claim(df, 'nsl_kdd_v2', 'dirichlet',
                      'NSL Dirichlet F1m drop (negative delta)',
                      'f1m_delta', 'negative'))

# Claim 5: NSL hybrid Brier worsens on most models
print(summarize_claim(df, 'nsl_kdd_v2', 'hybrid',
                      'NSL hybrid Brier change has CI excluding zero',
                      'brier_delta', 'any'))

# Claim 6: CIC dnn_5class_cw dramatic recovery (Brier improvement)
cic_dnn_cw_hyb = df[(df['dataset'] == 'cic_ids2017_v2') & (df['method'] == 'hybrid') & (df['model'] == 'dnn_5class_cw')]
if len(cic_dnn_cw_hyb):
    r = cic_dnn_cw_hyb.iloc[0]
    print(f'\nCIC dnn_5class_cw hybrid (catastrophic-model recovery):')
    print(f'  ΔBrier = {r["brier_delta_point"]:+.5f} [CI: {r["brier_delta_ci_lo"]:+.5f}, {r["brier_delta_ci_hi"]:+.5f}]')
    print(f'  ΔAcc = {r["acc_delta_point"]:+.4f} [CI: {r["acc_delta_ci_lo"]:+.4f}, {r["acc_delta_ci_hi"]:+.4f}]')
    print(f'  ΔF1m = {r["f1m_delta_point"]:+.4f} [CI: {r["f1m_delta_ci_lo"]:+.4f}, {r["f1m_delta_ci_hi"]:+.4f}]')
    print(f'  %flipped = {r["pct_flipped_point"]:.2f}% [CI: {r["pct_flipped_ci_lo"]:.2f}, {r["pct_flipped_ci_hi"]:.2f}]')

## 6. Per-dataset summary tables (formatted for paper)

In [ ]:
for dataset in DATASETS.keys():
    print(f'\n{"="*100}')
    print(f'{dataset} — Bootstrap 95% CIs')
    print("="*100)
    sub = df[df['dataset'] == dataset].copy()
    if len(sub) == 0:
        print('  (no data)')
        continue
    for method in METHODS.keys():
        sub_m = sub[sub['method'] == method]
        if len(sub_m) == 0:
            continue
        print(f'\n  Method: {method}')
        print(f'  {"Model":<20} {"ΔBrier (95% CI)":<30} {"ΔECE (95% CI)":<30} {"% flipped (95% CI)":<28}')
        print('  ' + '-'*108)
        for _, r in sub_m.iterrows():
            brier_str = f'{r["brier_delta_point"]:+.5f} [{r["brier_delta_ci_lo"]:+.5f}, {r["brier_delta_ci_hi"]:+.5f}]'
            ece_str = f'{r["ece_delta_point"]:+.5f} [{r["ece_delta_ci_lo"]:+.5f}, {r["ece_delta_ci_hi"]:+.5f}]'
            flip_str = f'{r["pct_flipped_point"]:.2f}% [{r["pct_flipped_ci_lo"]:.2f}, {r["pct_flipped_ci_hi"]:.2f}]'
            print(f'  {r["model"]:<20} {brier_str:<30} {ece_str:<30} {flip_str:<28}')

## 7. Commit

In [ ]:
os.chdir(REPO)
!git add notebooks/03d_calibration_bootstrap_cis.ipynb
!git add results/tables/calibration_bootstrap_cis.csv
!git status --short
!git commit -m 'Notebook 03d v2: Bootstrap 95% CIs on calibration metrics (B=1000, stratified)'
!git push origin main